### Step: Import Required Libraries

- **StateGraph**: Creates the workflow structure that connects different steps
- **START & END**: Mark the beginning and end of the workflow
- **TypedDict**: Defines the data structure passed between workflow steps
- **ChatGroq**: LLM model that generates responses
- **InMemorySaver**: Saves workflow history so it can be resumed later (persistence)

In [15]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

### Step: Set Up the LLM Model

- **Load environment variables**: Reads API keys from `.env` file
- **ChatGroq initialization**: Creates the LLM model that will generate responses
- **Input**: API key from environment
- **Output**: Ready-to-use model for the workflow

In [16]:
load_dotenv()
model = ChatGroq(model_name="llama-3.3-70b-versatile")

### Step: Define the State Structure (JokeState)

- **What it is**: A template that defines what data moves through the workflow
- **Data fields**:
  - `topic`: The subject the user wants a joke about
  - `joke`: The generated joke
  - `explanation`: Why the joke is funny
- **Why it matters**: Every node in the workflow reads and updates these fields

In [17]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explanation : str

### Step: Define the generate_joke Function

- **What it does**: Takes the topic from the state and generates a funny joke about it
- **Input**: `JokeState` containing the topic
- **Process**: 
  - Extracts the topic from state
  - Creates a prompt asking for a Gen Z humor joke
  - Calls the LLM (ChatGroq) to generate the joke
  - Stores the joke in the state
- **Output**: Updated state with the generated joke
- **Why it matters**: This is the first node in the workflow that creates the joke content

In [18]:
def generate_joke(state : JokeState) -> JokeState:
    topic = state['topic']

    prompt = f"Generate me funny humurous and gen z one joke on this topic {topic}"

    outline = model.invoke(prompt)

    state['joke'] = outline

    return state

### Step: Define the explain_joke Function

- **What it does**: Explains the joke in simple language and why it's funny
- **Input**: `JokeState` containing the topic and the generated joke
- **Process**: 
  - Extracts both the topic and the joke from state
  - Creates a prompt asking for a simple explanation of the joke
  - Calls the LLM to generate the explanation
  - Stores the explanation in the state
- **Output**: Updated state with the joke explanation
- **Why it matters**: This is the second node that adds context to make the joke more understandable

In [19]:
def explain_joke(state : JokeState) -> JokeState:
    topic = state['topic']
    joke = state['joke']

    prompt = f"Explain this joke {joke} and topic of joke was {topic}. Please Explain in simple language."

    explain = model.invoke(prompt)

    state['explanation'] = explain

    return state

### Step: Build the Workflow Graph

- **What it does**: Creates the workflow structure and connects all nodes together
- **Components**:
  - **StateGraph**: Creates a graph based on the JokeState structure
  - **add_node()**: Adds the two functions as nodes in the graph
  - **add_edge()**: Connects the nodes in sequence: START → generate_joke → explain_joke → END
  - **InMemorySaver**: Creates a checkpointer that saves workflow state to memory
  - **compile()**: Combines the graph and checkpointer into an executable workflow
- **Why it matters**: This establishes the workflow pipeline and enables persistence (saving state between executions)

In [20]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke" , generate_joke)
graph.add_node("explain_joke" , explain_joke)

graph.add_edge(START , "generate_joke")
graph.add_edge("generate_joke" , "explain_joke")
graph.add_edge("explain_joke" , END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

### Step: Execute the Workflow (First Run)

- **What it does**: Runs the complete workflow with a topic and saves the result
- **Key components**:
  - **config1**: Configuration with `thread_id: "1"` that uniquely identifies this workflow execution
  - **thread_id**: A string identifier that allows the same workflow to be resumed later with persistence
  - **input data**: `{'topic': 'momo'}` - the joke topic
  - **workflow.invoke()**: Executes all nodes in sequence (generate → explain) with the given input
- **Output**: The workflow generates a joke about "momo" and an explanation, all saved with thread_id "1"
- **Persistence concept**: The thread_id allows us to retrieve this exact workflow state later

In [21]:
config1 = {'configurable' : {"thread_id" : "1"}}

workflow.invoke({'topic' : 'momo'} , config= config1)

{'topic': 'momo',
 'joke': AIMessage(content='Here\'s one: Why did the momo go to therapy? Because it was feeling a little "wrapped up" in its own problems', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 50, 'total_tokens': 78, 'completion_time': 0.091990939, 'completion_tokens_details': None, 'prompt_time': 0.002235391, 'prompt_tokens_details': None, 'queue_time': 0.161816247, 'total_time': 0.09422633}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_4f6d808339', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019de310-b037-7440-a872-93c6e1d0854c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 28, 'total_tokens': 78}),
 'explanation': AIMessage(content='Let\'s break down the joke:\n\n**Joke:** Why did the momo go to therapy? Because it was feeling a little "wrapped up" in its own problems.\n\n**Explanation

### Step: Retrieve the Workflow State

- **What it does**: Fetches the saved workflow state for the thread with id "1"
- **How it works**:
  - Uses the same `config1` with `thread_id: "1"` to identify which workflow state to retrieve
  - Returns the complete state object containing all variables at the last checkpoint
  - The state includes: topic, joke, explanation, and other metadata
- **Persistence in action**: This demonstrates retrieving previously saved workflow progress
- **Use case**: Useful for resuming workflows, debugging, or checking intermediate results

In [22]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'momo', 'joke': AIMessage(content='Here\'s one: Why did the momo go to therapy? Because it was feeling a little "wrapped up" in its own problems', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 50, 'total_tokens': 78, 'completion_time': 0.091990939, 'completion_tokens_details': None, 'prompt_time': 0.002235391, 'prompt_tokens_details': None, 'queue_time': 0.161816247, 'total_time': 0.09422633}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_4f6d808339', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019de310-b037-7440-a872-93c6e1d0854c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 28, 'total_tokens': 78}), 'explanation': AIMessage(content='Let\'s break down the joke:\n\n**Joke:** Why did the momo go to therapy? Because it was feeling a little "wrapped up" in its own problem

### Step: Get Current State Data

- **What it does**: Actually executes the get_state call to retrieve the saved workflow state
- **Information returned**: 
  - Complete `JokeState` object with all fields (topic, joke, explanation)
  - Metadata about the checkpoint (when it was saved, which node it completed)
  - Other workflow execution details
- **Output format**: A state object that can be inspected
- **Next step**: We'll extract specific fields like the joke content from this state

In [23]:
workflow.get_state(config1).values['joke'].content

'Here\'s one: Why did the momo go to therapy? Because it was feeling a little "wrapped up" in its own problems'

### Step: Retrieve State at Specific Checkpoint

- **What it does**: Gets the workflow state at a specific point in time using a checkpoint_id
- **Key concepts**:
  - **thread_id: "1"**: Identifies which workflow run to look at
  - **checkpoint_id**: A unique identifier for a specific saved state/checkpoint
  - Each time the workflow completes a node, it creates a new checkpoint
- **Advanced persistence feature**: This allows you to:
  - Review intermediate workflow states
  - Time-travel through workflow execution history
  - Resume from any previous checkpoint if needed
- **Use case**: Debugging workflow execution or understanding how state evolved through different steps

In [24]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())